# 📊 Applying Machine-Learning Models on a Multi-Factor Equity Strategy

This notebook documents the implementation, training, and evaluation of a machine learning model for dynamically weighting momentum and profitability within the framework of a factor investing strategy.

---

🗂️ Table of Contents
1. 🔧 Setup & Parameters
2. 💾 Data
3. 🏅 Calculate Stock Ranks (for each factor)
4. 🧠 Model Training
5. 🚀 Model Application
6. 📉 Backtest Calculation
7. 📊 Results Visualization
    7.1 📉 Choice of Training Period
    7.2 📉 Choice of Feature Variables
    7.3 📉 Choice of Machine Learning Model


## Abstract

Accurately forecasting directional movements and optimally constructing equity portfolios are central goals in systematized investing. Factor investing exploits consistent signals such as momentum and profitability to achieve durable excess returns. In this project, we apply modern machine learning models—including Random Forest, XGBoost, LightGBM, and CatBoost—to dynamically combine multiple factor signals for stock selection and portfolio allocation.

For each model, we demonstrate the impact of training period, feature selection, and hyperparameter tuning on predictive accuracy. The configuration delivering the highest validation accuracy is selected for out-of-sample vectorized backtests, where strategy returns are simulated and benchmarked against traditional static blends (such as equal-weighted factors) and the market index. The adaptive ML-driven approach allocates portfolio weights monthly, taking long positions in stocks with strong combined factor scores and short or underweights elsewhere.

All stages—from data ingestion and factor score computation, through feature engineering, model training, and performance evaluation—are streamlined for clarity and reproducibility. Portfolio evaluation employs standard metrics: annualized return, volatility, Sharpe ratio, and maximum drawdown. Results indicate that the machine learning-based dynamic factor weighting can outperform naïve static approaches and provides a flexible, extensible framework for systematic investing. Further extensions may incorporate additional factors or alternative learning algorithms for enhanced strategy robustness

## 🔧 1. Setup & Parameters

Basic configuration and parameters. Clearly separating the training and test data ensures a valid assessment of model predictions.

In [1]:
import pandas as pd
from ML_Multifactor import run_ml_backtests

In [2]:
params_ = {'use_pickle_data': True,
           'update_factor_scores': False,
           'price_frequency_str': 'D',
           'price_frequency_num': 252,
           'training_start_period': pd.Timestamp('2007-01-31'),
           'training_end_period': pd.Timestamp('2012-12-31'),
           'test_start_period': pd.Timestamp('2013-01-01'),
           'test_end_period': pd.Timestamp('2025-03-20'),
           'ml_training_factors': ['mom_roe_corr', 'past_mom_return', 'past_roe_return',
                                   'past_index_return', 'past_index_vola', 'rf', 'vix'],
           'relevant_factors': ['mom', 'roe'],
           'ml_model': 'RandomForestRegressor'}

## 📦 2. Data

The `GetData` class is designed to manage the entire data loading and preprocessing pipeline required for factor-based equity analysis. Its main objective is to provide a simple, reproducible, and efficient framework to access all relevant financial data—such as historical stock prices, index weights, factor values (e.g., ROE), risk-free rates, and volatility indices—either from raw Excel files or from pickled pre-processed files for faster repeated execution.

- **Initialization:** The class is configured using a parameter dictionary allowing flexible specification of price frequency (e.g., daily, monthly), data source (raw vs. pickled), and any other relevant settings.
- **`get_data()`** loads and organizes all datasets into consistent pandas DataFrames and Series, handling data mapping, resampling, reindexing, and missing value imputation where necessary. Outputs are stored as class attributes for downstream analysis.
- **`get_descriptive_stats()`** computes key statistics such as dataset dimensions, missing data rates, and annualized return characteristics—helping validate data quality before modeling and backtesting.

The goal is to abstract all tedious data handling tasks into one robust object, enabling a clean separation between data preparation and subsequent modeling steps.


## 🏅 3. Calculate Stock Scores (for fach factor)

The `GetStockScores` class is responsible for calculating, ranking, and managing factor-based scores for each stock in the investment universe. Its core objective is to translate raw price and fundamental data into standardized factor metrics—such as momentum, volatility, and Return on Equity (ROE)—which can be used in systematic portfolio construction and machine learning models.

- **Initialization:** The class takes as input the preprocessed data from a `GetData` object as well as user-defined parameters (such as frequency settings and choice of factors).
- **Factor Calculation:** Using rolling windows, the class computes:
    - 12-month price momentum (relative price change vs. one year ago)
    - 12-month price volatility (rolling annualized standard deviation)
    - 12-month mean ROE (average profitability)
- **Ranking:** For each factor and each date, stocks are assigned to quintiles based on their relative factor values. This results in standardized ranks (1-5), allowing for robust cross-sectional analysis and selection.
- **Caching:** To improve efficiency, scores can be quickly loaded from disk if previously computed, or recalculated from scratch for full data refresh.
- **Outputs:** All scores and ranks are stored as attributes, enabling seamless downstream integration with portfolio generation, machine learning models, and strategy evaluation.

By centralizing factor score computation and ranking, `GetStockScores` serves as a foundational building block for data-driven equity screening, backtesting, and allocation workflows.

## 🧠 4. Model Training

The `TrainMLModel` class orchestrates the process of training supervised machine learning regressors to dynamically determine optimal blend weights between equity factors (e.g., momentum and ROE). The main goal is to learn from historical factor scores, market variables, and index information how to allocate between these signals to maximize forward-looking portfolio performance.

- **Initialization:** The class takes precomputed factor scores (from `GetStockScores`), raw and processed market data (from `GetData`), and a flexible parameter dictionary (specifying training time window, selected factors, and model type).
- **Feature Engineering and Target Calculation:** For each eligible training period, the class constructs feature vectors describing the market environment, factor values, and asset characteristics. It simultaneously determines the optimal factor combination weight (target) through an exhaustive search maximizing the out-of-sample return in subsequent periods.
- **Model Fitting:** Various tree-based regressors can be specified—including Random Forest, LightGBM, XGBoost, and CatBoost—which are trained to predict the optimal blend weight from the engineered features.
- **Evaluation:** The class evaluates model performance using standard regression metrics (e.g., R², MSE, MAE), extracts model-specific feature importances, and computes univariate statistics (F-score, p-value) for all features. These diagnostics help assess the model’s predictive power and interpret key drivers of optimal portfolio blend weights.
- **Output:** After training, the class exposes the fitted model and all performance analytics as attributes, making them readily available for downstream blending, portfolio construction, and strategy analysis.

Overall, `TrainMLModel` integrates data, factors, and advanced machine learning to enable state-of-the-art, data-adaptive portfolio allocation in systematic equity research.

## 🚀 5. Model Application

The `ApplyMLModel` class serves as the operational bridge between a previously trained machine learning blending model and live or recent market and factor data. Its purpose is to generate time-varying portfolio construction signals—by dynamically integrating factor ranks and market features—based on the predictions provided by the supervised learning model.

- **Initialization:** The class takes as input the processed datasets (`GetData`), factor scores (`GetStockScores`), the trained machine learning model (`TrainMLModel`), and a configuration dictionary. Key factor ranks, rolling scores, and market variables are retained as attributes for subsequent weighting.
- **Factor Weight Inference:** For each period, a feature vector is created from up-to-date market and factor data; the trained model predicts the optimal blend weight for combining factors (e.g., momentum and ROE). Alternative static blends (such as 50/50 or average weight) are also supported for robust benchmarking.
- **Portfolio Construction:** Based on the predicted weights and factor ranks, stock-level portfolio allocations are computed for each strategy. These include: machine learning-based, equal-weighted, average-weight, and index benchmark portfolios. Weights are rescaled for full allocation and consistency across dates.
- **Outputs:** All portfolio weights, factor blending summaries, and performance diagnostics are stored as attributes for downstream analysis and backtesting. This enables transparent comparison of dynamic ML-driven portfolios versus traditional fixed-weight or index-based allocations.

By encapsulating model inference, weighting logic, and flexible portfolio generation, `ApplyMLModel` empowers seamless deployment and evaluation of adaptive, data-driven investment strategies in systematic equity research.

## 📉 6. Backtest Calculation

The `CalculateBacktest` class is built to rigorously evaluate the historical performance of different portfolio construction strategies—such as those based on machine learning-driven factor blends, equal weighting, and index benchmarks. Its main objective is to simulate realistic portfolio returns under various allocation schemes and compute key metrics for performance comparison.

- **Initialization:** The class is initialized with market return data (`GetData`), a configuration dictionary (including start/end periods and rebalancing frequency), and precomputed strategy weights from the `ApplyMLModel` class. It stores all inputs and prepares placeholders for results.
- **Backtest Execution:** The `run_backtest()` method simulates monthly (or custom) rebalancing portfolios over the defined test period. For each rebalancing date, realized daily returns are computed by applying strategy weights to asset returns, updating portfolio values through time.
- **Performance Computation:** After returns are generated for each strategy, the class calculates annualized performance metrics—including average return, volatility, Sharpe ratio, and maximum drawdown—allowing for standardized comparison and risk assessment.
- **Outputs:** Both portfolio return time series and aggregated performance statistics are retained as attributes, making them readily available for reporting, visualization, and further analysis.

By systematizing return calculation and evaluation across dynamic and benchmark strategies, `CalculateBacktest` provides robust quantitative evidence for the effectiveness of machine learning-driven portfolio allocation in systematic investing workflows.

## 📊 7. Results Visualization

The backtest results are visualized to facilitate intuitive assessment and comparison of portfolio performance over time.

- 📈 **Cumulative Returns Plot:**
  Line charts display the growth of $1 invested in each strategy, highlighting differences in compounding and drawdowns.

- 🗃️ **Risk & Return Tables:**
  Tabular summaries show annualized returns, volatility, max drawdown, Sharpe and Sortino ratios, and other key performance metrics for each strategy.

- 🔄 **Dynamic Factor Weighting:**
  Time-series plots illustrate how the ML model's factor weights evolve through different market regimes.

- 🆚 **Benchmark Comparison:**
  Visuals compare the ML-driven strategy against static and index benchmarks for easy interpretation.

- 🧩 **Additional Insights:**
  Optional plots may include rolling statistics, turnover, contribution by factor, and feature importance (if available).

**Outcome:**
The visualizations provide clarity on how the strategies performed, how risks were managed, and how dynamic weighting impacted results—supporting informed conclusions and further research.

---

## 7.1. Choice of Training Period

Selecting an appropriate training period is crucial for robust model performance and meaningful backtest results.

- 📆 **Rolling vs. Expanding Window:**
  - *Rolling Window:* Uses a fixed-length lookback period (e.g., last 36 months) up to each rebalance date. This helps the model adapt to changing market conditions and mitigates lookahead bias.
  - *Expanding Window:* Starts at a fixed point in the past and progressively adds more data with each rebalance. This leverages all available information but may underweight recent dynamics.

- 🕰️ **Period Length Considerations:**
  - The window should be long enough to capture several different market regimes and reduce noise, but not so long that the model becomes unresponsive to recent changes.
  - Common choices range from 2 to 5 years, depending on data frequency, anticipated factor cyclicality, and desired model adaptability.

- ⚠️ **Overfitting and Data Leakage:**
  - Training strictly uses information available prior to each rebalance date.
  - Validation and test sets should always be forward-looking (i.e., after the training period).

- 🧪 **Robustness Checks:**
  - Analysis may include sensitivity tests using alternative window lengths or split points to ensure results are not overly dependent on arbitrary period choices.

**Outcome:**
A thoughtful training period design balances stability, adaptability, and realism—laying the foundation for meaningful out-of-sample evaluation.

---

In [ ]:
# 2. Set model specifications
model_definitions = [
    dict(name="Model 1",
         training_end_period='2009-12-31',
         test_start_period='2010-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return', 'past_index_return', 'past_index_vola', 'rf', 'vix']),
    dict(name="Model 2",
         training_end_period='2010-12-31',
         test_start_period='2011-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return', 'past_index_return', 'past_index_vola', 'rf', 'vix'])
]

# 3. Run models
results = run_ml_backtests(model_definitions, params_)

🚀 Start Model 1


In [ ]:
# Display goodness of model
df = pd.concat([res['GoodnessOfModel'] for res in results])
display(df)

In [ ]:
# Excess Return (noch zu stylen)
df = pd.concat([res['ExcessReturns'] for res in results])
display(df)

In [ ]:
# Display Importance
df = pd.concat([res['FeatureImportance'] for res in results])
display(df)

In [ ]:
# Display Feature Stats (p.Values)
df = pd.concat([res['FeatureStats'] for res in results])
display(df)

In [ ]:
# Display Factor Weighting over Time
print("Please print factor weighting over time")

In [ ]:
# 2. Set model specifications
model_definitions = [
    dict(name="Model 4",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return']),
    dict(name="Model 5",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['past_mom_return', 'past_roe_return']),
    dict(name="Model 6",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf', 'vix']),
    dict(name="Model 7",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['past_index_return', 'past_index_vola']),
    dict(name="Model 8",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return', 'past_index_return', 'past_index_vola', 'rf', 'vix'],
         ml_model='RandomForestRegressor'),
    dict(name="Model 9",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return', 'past_index_return', 'past_index_vola', 'rf', 'vix'],
         ml_model='lightgbm'),
    dict(name="Model 10",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return', 'past_index_return', 'past_index_vola', 'rf', 'vix'],
         ml_model='xgboost'),
    dict(name="Model 11",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['mom_roe_corr', 'past_mom_return', 'past_roe_return', 'past_index_return', 'past_index_vola', 'rf', 'vix'],
         ml_model='catboost'),
    dict(name="Model 12",
         training_end_period='2014-12-31',
         test_start_period='2015-01-01',
         ml_training_factors=['rf', 'vix'],
         ml_model='xgboost')
]